In [1]:
import numpy as np
from pydrake.systems.framework import DiagramBuilder, LeafSystem
from pydrake.examples import PendulumPlant
from pydrake.systems.analysis import Simulator
from pydrake.systems.primitives import LogVectorOutput
from dataclasses import dataclass

In [2]:
@dataclass
class Config:
    k: float  = 1.0

config = Config()

In [3]:
from pydrake.geometry import StartMeshcat, MeshcatVisualizer, SceneGraph
from pydrake.examples import PendulumGeometry

meshcat = StartMeshcat()    

INFO:drake:Meshcat listening for connections at http://localhost:7000


In [5]:
class EnergyController(LeafSystem):
    def __init__(self, config: Config, m: float, l: float, g: float):
        super().__init__()
        self.k = config.k 
        self.m = m 
        self.l = l
        self.g = g
        self.E_d = m *g * l
        self.DeclareVectorInputPort('state',2)
        self.DeclareVectorOutputPort('u', 1, self.CalcTorque)
    
    def CalcTorque(self, context, output):
        
        theta, theta_dot = self.get_input_port(0).Eval(context)
        E = 0.5 * self.m * (self.l ** 2) * (theta_dot ** 2) - self.m * self.g * self.l * np.cos(theta)

        u = - self.k * theta_dot * (E - self.E_d)

        output.SetAtIndex(0 ,u)





In [ ]:
builder = DiagramBuilder()
pendulum = builder.AddSystem(PendulumPlant())

print("input: ", pendulum.get_input_port(0).get_name())
print("output:", pendulum.get_output_port(0).get_name())

context  = pendulum.CreateDefaultContext()
params = pendulum.get_parameters(context)
print("m =", params.mass(), " l =", params.length(), " b =", params.damping(), " g =", params.gravity())

controller = builder.AddSystem(EnergyController(config, m=params.mass(), l=params.length(), g=params.gravity()))

builder.Connect(pendulum.get_output_port(0), controller.get_input_port(0))
builder.Connect(controller.get_output_port(0), pendulum.get_input_port(0))

logger = LogVectorOutput(pendulum.get_output_port(0), builder)


scene_graph = builder.AddSystem(SceneGraph())
PendulumGeometry.AddToBuilder(builder, pendulum.get_state_output_port(), scene_graph)
visualizer = MeshcatVisualizer.AddToBuilder(builder, scene_graph, meshcat)

diagram = builder.Build()

simulator = Simulator(diagram)
context = simulator.get_mutable_context()
context.SetContinuousState([0.0, 0.1]) 

simulator.set_target_realtime_rate(1.0)  
visualizer.StartRecording()
simulator.AdvanceTo(10.0)
visualizer.PublishRecording()             